
# Neural Network: Beer-Lambert Law for Absorbance
## Chem 190/290, Spring 2026, Liang Shi


A student has done the absorption spectrum experiment, and saved the data in the text file "concentration_intensity.txt". The file has two columns: the first column is concentration, and the second is transmitted light intensity. 


In [ ]:
# install scikit-learn
# %pip install scikit-learn

# Bring in numpy(for numerical operations) and matplotlib.pyplot (for graphics)
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor

In [ ]:
%matplotlib inline

## 1. Data loading and pre-processing

In [ ]:
# Load the data
data = np.loadtxt("concentration_intensity.txt")

# Split into columns
concentration0 = data[:, 0]
intensity0 = data[:, 1]

print(concentration0)
print(intensity0)

In [ ]:
# Reference intensity (zero concentration)
I0 = intensity0[concentration0 == 0][0]

# Compute absorbance
absorbance0 = -np.log10(intensity0 / I0)

print(absorbance0)

In [ ]:
# Plot
plt.figure()
plt.scatter(concentration0, absorbance0)
plt.xlabel("Concentration")
plt.ylabel("Absorbance")
plt.title("Absorbance vs Concentration")
plt.show()

## 2. Linear Regression using Normal Equation

In [ ]:
# remove the zero concentration data from our fitting
absorbance = absorbance0[1:]
concentration = concentration0[1:]

y = absorbance.reshape(-1, 1) # make the label vector
X = np.column_stack((np.ones(len(concentration)), concentration)) # make the data matrix

print("data matrix:")
print(X)
print("label vector:")
print(y)

# normal equation
theta = np.linalg.inv(X.T @ X) @ X.T @ y
print("theta values:")
print(theta)

# Predicted values at training X
y_pred = X @ theta
print("Predicted values:")
print(y_pred)

# Compute MSE and MAE
residuals = y - y_pred
mse = np.mean(residuals**2)
mae = np.mean(np.abs(residuals))
print("Mean Squared Error (MSE):", mse)
print("Mean Absolute Error (MAE):", mae)

# Plot
plt.scatter(concentration, y, label="Data")
plt.plot(concentration, y_pred, label="Normal Equation Fit")
plt.xlim(-0.05, 1.05)
plt.ylim(-0.05, 2.55)
plt.legend()
plt.show()

## 3. Neural Network with Linear Activation Function

In [ ]:
# Ensure y is 1D
X = concentration.reshape(-1, 1)
y = absorbance

# Neural network with 1 hidden layer, 1 neuron, linear activation
# hidden_layer_sizes=(1,) → 1 neuron
# activation='identity' → linear
# solver='lbfgs' → deterministic solver, good for small datasets
model = MLPRegressor(hidden_layer_sizes=(1,),
                     activation='identity',
                     solver='lbfgs',
                     alpha=0,            # no regularization
                     max_iter=100,
                     random_state=0)

# for model.fit
# X: ndarray or sparse matrix of shape (n_samples, n_features) for input
# y: ndarray of shape (n_samples,) or (n_samples, n_outputs) for output
model.fit(X, y)

# Predicted values at training X
y_train_pred = model.predict(X)
print("Predicted values at training X:")
print(y_train_pred)

# Compute MSE and MAE
residuals = y - y_pred
mse = np.mean(residuals**2)
mae = np.mean(np.abs(residuals))
print("Mean Squared Error (MSE):", mse)
print("Mean Absolute Error (MAE):", mae)

# Plot
plt.scatter(concentration, absorbance, label="Data")
plt.plot(concentration, y_train_pred, label="Neural Network (1 hidden neuron, linear)")
plt.xlim(-0.05, 1.05)
plt.ylim(-0.05, 2.55)
plt.legend()
plt.show()

In [ ]:
# change the solver in MLPRegressor to adam

In [ ]:
# use PyTorch instead of scikit-learn for NN

import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# Set random seed for reproducibility
# -----------------------------
torch.manual_seed(2026)
np.random.seed(2026)

# -----------------------------
# Prepare data
# -----------------------------
# Convert concentration and absorbance to PyTorch tensors
X = torch.tensor(concentration.reshape(-1, 1), dtype=torch.float32)
y = torch.tensor(absorbance.reshape(-1, 1), dtype=torch.float32)

# -----------------------------
# Define the neural network
# -----------------------------
class LinearNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Single hidden neuron, linear activation, output 1
        self.hidden = nn.Linear(1, 1, bias=True)  # bias included
        self.output = nn.Identity()  # linear activation

    def forward(self, x):
        x = self.hidden(x)
        x = self.output(x)
        return x

model = LinearNN()

# -----------------------------
# Define optimizer and loss
# -----------------------------
optimizer = optim.Adam(model.parameters(), lr=0.1)
mse_loss_fn = nn.MSELoss()
mae_loss_fn = nn.L1Loss()

# -----------------------------
# Training loop
# -----------------------------
epochs = 500
mse_list = []
mae_list = []

for epoch in range(epochs):
    model.train()
    
    # Forward pass
    y_pred = model(X)
    
    # Compute MSE for backprop
    loss = mse_loss_fn(y_pred, y)
    
    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # Record errors
    mse_list.append(mse_loss_fn(y_pred, y).item())
    mae_list.append(mae_loss_fn(y_pred, y).item())

# -----------------------------
# Final prediction
# -----------------------------
model.eval()
with torch.no_grad():
    y_train_pred = model(X).numpy()

print("Predicted values at training X:")
print(y_train_pred)

# -----------------------------
# Print final MSE and MAE
# -----------------------------
final_mse = mse_loss_fn(torch.tensor(y_train_pred), y).item()
final_mae = mae_loss_fn(torch.tensor(y_train_pred), y).item()
print(f"Final MSE: {final_mse:.6f}")
print(f"Final MAE: {final_mae:.6f}")

# -----------------------------
# Plot fit
# -----------------------------
plt.scatter(concentration, absorbance, label="Data")
plt.plot(concentration, y_train_pred, label="PyTorch Linear NN", color='orange')
plt.xlim(-0.05, 1.05)
plt.ylim(-0.05, 2.55)
plt.legend()
plt.show()

# -----------------------------
# Two-panel error plot (log scale)
# -----------------------------
fig, axes = plt.subplots(2, 1, figsize=(6, 8), sharex=True)

axes[0].plot(range(1, epochs+1), mse_list)
axes[0].set_yscale("log")
axes[0].set_ylabel("MSE (log scale)")
axes[0].set_title("Training Error vs Epoch")
axes[0].grid(True, which='both', linestyle='--', alpha=0.5)

axes[1].plot(range(1, epochs+1), mae_list)
axes[1].set_yscale("log")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MAE (log scale)")
axes[1].grid(True, which='both', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# -----------------------------
# Parity plot: predicted vs actual (square)
# -----------------------------
plt.figure(figsize=(6,6))
plt.scatter(absorbance, y_train_pred, color='green')
min_val = min(absorbance.min(), y_train_pred.min())
max_val = max(absorbance.max(), y_train_pred.max())
plt.plot([min_val, max_val],
         [min_val, max_val],
         'r--', label='Perfect Prediction')
plt.xlabel("Actual y")
plt.ylabel("Predicted y")
plt.title("Parity Plot")
plt.legend()
plt.grid(alpha=0.3)
plt.xlim(min_val, max_val)
plt.ylim(min_val, max_val)
plt.axis('square')  # make it square
plt.show()

# -----------------------------
# Compute correlation coefficient and R^2
# -----------------------------
import numpy as np

# Flatten arrays just in case
y_true = absorbance.flatten()
y_pred = y_train_pred.flatten()

# Pearson correlation coefficient
r_matrix = np.corrcoef(y_true, y_pred)
r = r_matrix[0, 1]

# Coefficient of determination
r2 = r**2

print(f"Correlation coefficient (r): {r:.6f}")
print(f"R-squared (r^2): {r2:.6f}")

## 4. Neural Network with Nonlinear Activation Function

In [ ]:
# still use PyTorch, but now change the activation function from linear to ReLU
# namely, linear --> ReLU
# you might have to do something else, let's explore this.